In [ ]:
# Install required packages
!pip install -q transformers datasets torch

## Environment Setup

In [ ]:
# Import libraries
import sys
import os
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader
import torch
import time
from datetime import datetime

# Verify installations
try:
    import torch
    import transformers
    import datasets
    print("✅ PyTorch version:", torch.__version__)
    print("✅ Transformers version:", transformers.__version__)
    print("✅ Datasets version:", datasets.__version__)
except ImportError as e:
    print("❌ Import error:", e)

✅ PyTorch version: 2.8.0+cu126
✅ Transformers version: 4.57.1
✅ Datasets version: 4.0.0


In [ ]:
# Check GPU availability
print("\n" + "="*60)
print("SYSTEM INFORMATION")
print("="*60)
print(f"\n🔍 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔍 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🔍 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("🔍 Running on CPU")
print(f"\n⏰ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


SYSTEM INFORMATION

🔍 CUDA available: True
🔍 GPU: Tesla T4
🔍 GPU Memory: 15.83 GB

⏰ Current time: 2025-11-16 17:46:26


In [ ]:
print("="*60)
print("STEP 1: LOADING DATASET IN STREAMING MODE")
print("="*60)
print(f"\n⏰ Started at: {datetime.now().strftime('%H:%M:%S')}")
print("\n📥 Loading Yelp Reviews in streaming mode...")
print("   (Processing ALL 650K reviews without loading into memory!)\n")

start_time = time.time()

# Load Yelp Reviews with streaming=True
stream_dataset = load_dataset(
    "yelp_review_full",
    split="train",
    streaming=True  # ← KEY: Enables streaming!
)

load_time = time.time() - start_time

print("✅ Streaming dataset loaded!")
print(f"\n⏱️  Loading time: {load_time:.2f} seconds")
print(f"💾 Memory used: ~0 MB (data not loaded yet!)")
print(f"\n💡 Dataset size: 650K reviews")
print(f"💡 Can iterate through ALL reviews without memory issues!")

STEP 1: LOADING DATASET IN STREAMING MODE

⏰ Started at: 17:46:45

📥 Loading Yelp Reviews in streaming mode...
   (Processing ALL 650K reviews without loading into memory!)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

✅ Streaming dataset loaded!

⏱️  Loading time: 3.33 seconds
💾 Memory used: ~0 MB (data not loaded yet!)

💡 Dataset size: 650K reviews
💡 Can iterate through ALL reviews without memory issues!


In [ ]:
# Peek at first example
print("\n" + "="*60)
print(" PEEKING AT FIRST REVIEW")
print("="*60)
print("\n Fetching first review from stream...\n")

first_example = next(iter(stream_dataset))

print(f"Rating: {'⭐' * first_example['label']} stars")
print(f"\nReview text (first 400 characters):\n")
print(first_example['text'][:400])
print("\n...")
print(f"\n Full review length: {len(first_example['text'])} characters")


 PEEKING AT FIRST REVIEW

 Fetching first review from stream...

Rating: ⭐⭐⭐⭐ stars

Review text (first 400 characters):

dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first.

...

 Full review length: 534 characters


In [ ]:
print("="*60)
print("STEP 2: TOKENIZER SETUP")
print("="*60)
print("\n🔧 Loading GPT-2 tokenizer...\n")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Set pad token

print("✅ Tokenizer loaded successfully!")
print(f"\n📊 Vocabulary size: {tokenizer.vocab_size:,} tokens")
print(f"🔑 EOS token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")
print(f"🔑 PAD token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")

STEP 2: TOKENIZER SETUP

🔧 Loading GPT-2 tokenizer...



tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

✅ Tokenizer loaded successfully!

📊 Vocabulary size: 50,257 tokens
🔑 EOS token: '<|endoftext|>' (ID: 50256)
🔑 PAD token: '<|endoftext|>' (ID: 50256)


In [ ]:
print("="*60)
print("STEP 3: DEFINING ROLLING BUFFER FUNCTION")
print("="*60)

block_size = 128  # Sequence length for training

def group_texts_streaming(dataset_iter, block_size):
    """
    Generator that yields fixed-length blocks from a streaming dataset.

    Uses a rolling buffer to accumulate tokens across reviews:
    1. Add tokens from current review to buffer
    2. Yield complete blocks when buffer is full
    3. Keep remainder for next iteration

    Args:
        dataset_iter: Iterator over tokenized examples
        block_size: Target length for each block

    Yields:
        Dict with 'input_ids' and 'attention_mask' of length block_size
    """
    buffer = []  # Rolling buffer to accumulate tokens

    for example in dataset_iter:
        # Add tokens from this review to buffer
        buffer.extend(example["input_ids"])

        # Yield complete blocks whenever buffer is large enough
        while len(buffer) >= block_size:
            chunk = buffer[:block_size]
            buffer = buffer[block_size:]  # Keep remainder

            yield {
                "input_ids": chunk,
                "attention_mask": [1] * block_size
            }

    # Note: Leftover tokens at the end are typically discarded
    # (For infinite streaming, this rarely happens)

print(f"\n✅ Rolling buffer function defined")
print(f"\n🎯 Configuration:")
print(f"   - Block size: {block_size} tokens")
print(f"   - Buffer: Dynamic (grows and shrinks)")
print(f"   - Memory: O(block_size) - constant!")

STEP 3: DEFINING ROLLING BUFFER FUNCTION

✅ Rolling buffer function defined

🎯 Configuration:
   - Block size: 128 tokens
   - Buffer: Dynamic (grows and shrinks)
   - Memory: O(block_size) - constant!


In [ ]:
print("="*60)
print("STEP 4: CREATING ITERABLE DATASET")
print("="*60)

class StreamingLMIterableDataset(IterableDataset):
    """
    PyTorch IterableDataset wrapper for streaming language modeling data.

    This class:
    1. Takes a streaming HuggingFace dataset
    2. Tokenizes text on-the-fly
    3. Groups into fixed-length blocks using rolling buffer
    """
    def __init__(self, hf_iterable_dataset, tokenizer, block_size):
        self.dataset = hf_iterable_dataset
        self.tokenizer = tokenizer
        self.block_size = block_size

    def __iter__(self):
        # Create token stream by tokenizing each text example
        tokenized_stream = self.dataset.map(
            lambda examples: self.tokenizer(examples["text"]),
            batched=True
        )

        # Yield fixed-length blocks from the token stream
        return group_texts_streaming(tokenized_stream, self.block_size)

# Create the iterable dataset
grouped_iterable_dataset = StreamingLMIterableDataset(
    stream_dataset,
    tokenizer,
    block_size
)

print("\n✅ Streaming IterableDataset created!")
print(f"\n📊 Configuration:")
print(f"   - Dataset: Yelp Reviews (streaming)")
print(f"   - Tokenizer: GPT-2")
print(f"   - Block size: {block_size}")
print(f"   - Type: IterableDataset (lazy loading)")

STEP 4: CREATING ITERABLE DATASET

✅ Streaming IterableDataset created!

📊 Configuration:
   - Dataset: Yelp Reviews (streaming)
   - Tokenizer: GPT-2
   - Block size: 128
   - Type: IterableDataset (lazy loading)


In [ ]:
print("="*60)
print("STEP 5: CREATING DATALOADER")
print("="*60)

def collate_fn(batch):
    """
    Collate function for DataLoader.
    Converts list of dicts to batched tensors.
    """
    input_ids = torch.tensor(
        [ex["input_ids"] for ex in batch],
        dtype=torch.long
    )
    attention_mask = torch.tensor(
        [ex["attention_mask"] for ex in batch],
        dtype=torch.long
    )
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.clone()  # For causal LM
    }

# Create streaming DataLoader
batch_size = 8
train_loader = DataLoader(
    grouped_iterable_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    num_workers=0  # Important: Don't use workers with IterableDataset
)

print("\n✅ Streaming DataLoader created!")
print(f"\n📊 Configuration:")
print(f"   - Batch size: {batch_size}")
print(f"   - Shuffle: No (streaming datasets can't shuffle)")
print(f"   - Workers: 0 (streaming doesn't need workers)")
print(f"\n💡 Note: Can iterate indefinitely without running out of memory!")

STEP 5: CREATING DATALOADER

✅ Streaming DataLoader created!

📊 Configuration:
   - Batch size: 8
   - Shuffle: No (streaming datasets can't shuffle)
   - Workers: 0 (streaming doesn't need workers)

💡 Note: Can iterate indefinitely without running out of memory!


In [ ]:
print("="*60)
print("STEP 6: INSPECTING STREAMING BATCHES")
print("="*60)
print("\n🔍 Fetching 3 sample batches from stream...\n")

start_time = time.time()

for i, batch in enumerate(train_loader):
    print(f"\n{'─'*60}")
    print(f"📦 Batch {i}:")
    print(f"{'─'*60}")
    print(f"   input_ids shape:     {batch['input_ids'].shape}")
    print(f"   attention_mask shape: {batch['attention_mask'].shape}")
    print(f"   labels shape:        {batch['labels'].shape}")
    print(f"   dtype:               {batch['input_ids'].dtype}")

    # Show decoded text for first batch
    if i == 0:
        print(f"\n   📝 First sequence (decoded):")
        decoded = tokenizer.decode(batch['input_ids'][0])
        print(f"   {decoded[:300]}...")

    if i == 2:  # Show 3 batches
        break

elapsed = time.time() - start_time

print(f"\n\n{'='*60}")
print("✅ BATCHES FETCHED SUCCESSFULLY")
print(f"{'='*60}")
print(f"⏱️  Time to fetch 3 batches: {elapsed:.2f} seconds")
print(f"📊 Batch shape: ({batch_size}, {block_size})")

STEP 6: INSPECTING STREAMING BATCHES

🔍 Fetching 3 sample batches from stream...



Token indices sequence length is longer than the specified maximum sequence length for this model (1132 > 1024). Running this sequence through the model will result in indexing errors



────────────────────────────────────────────────────────────
📦 Batch 0:
────────────────────────────────────────────────────────────
   input_ids shape:     torch.Size([8, 128])
   attention_mask shape: torch.Size([8, 128])
   labels shape:        torch.Size([8, 128])
   dtype:               torch.int64

   📝 First sequence (decoded):
   dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens...

────────────────────────────────────────────────────────────
📦 Batch 1:
────────────────────────────────────────────────────────────
   input_ids shape:     torch.Size([8, 128])
   attention_mask shape: torch.Size([8, 128])
   labels shape:        torch.Size([8, 128])
   dtype:               torch.int64

────────────────────────────────────────────────

In [ ]:
print("\n\n" + "="*60)
print("🎉 LAB 2 COMPLETE - STREAMING PIPELINE")
print("="*60)

print(f"\n📊 LAB 2 STATISTICS:")
print(f"   {'─'*50}")
print(f"   Dataset:              Yelp Reviews (FULL - 650K)")
print(f"   Loading approach:     Streaming (on-the-fly)")
print(f"   Sequence length:      {block_size} tokens")
print(f"   Batch size:           {batch_size}")
print(f"   Vocabulary size:      {tokenizer.vocab_size:,}")

print(f"\n💾 MEMORY CHARACTERISTICS:")
print(f"   {'─'*50}")
print(f"   Approach:             Streaming (lazy loading)")
print(f"   Memory usage:         ~50-100 MB")
print(f"   Max dataset size:     UNLIMITED")
print(f"   Can iterate:          Indefinitely")


print(f"\n💡 WHEN TO USE STREAMING:")
print(f"   {'─'*50}")
print(f"   ✅ Dataset > 10 GB")
print(f"   ✅ Limited RAM available")
print(f"   ✅ Training on cloud with expensive storage")
print(f"   ✅ Continuous/online learning scenarios")
print(f"   ✅ Production ML systems")




🎉 LAB 2 COMPLETE - STREAMING PIPELINE

📊 LAB 2 STATISTICS:
   ──────────────────────────────────────────────────
   Dataset:              Yelp Reviews (FULL - 650K)
   Loading approach:     Streaming (on-the-fly)
   Sequence length:      128 tokens
   Batch size:           8
   Vocabulary size:      50,257

💾 MEMORY CHARACTERISTICS:
   ──────────────────────────────────────────────────
   Approach:             Streaming (lazy loading)
   Memory usage:         ~50-100 MB
   Max dataset size:     UNLIMITED
   Can iterate:          Indefinitely

💡 WHEN TO USE STREAMING:
   ──────────────────────────────────────────────────
   ✅ Dataset > 10 GB
   ✅ Limited RAM available
   ✅ Training on cloud with expensive storage
   ✅ Continuous/online learning scenarios
   ✅ Production ML systems


In [ ]:
,
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4